**Choco Crunch Analytics — SQL Table Design**

**Retrieve the Final Dataset**

The feature-engineered DataFrame `df_main` is retrieved from the previous notebook using the IPython `%store` mechanism.

This DataFrame contains the product information, nutritional features, and derived metrics that will be organized into relational tables in PostgreSQL.

In [1]:
%store -r df_main

**Connect to PostgreSQL**

A connection is established with the `CHOCO_CRUNCH` PostgreSQL database using `psycopg2`.

A cursor is created to execute SQL statements from Python. This connection will be used to create the database tables and insert the feature-engineered data.

In [2]:
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

# Create a cursor object
cur = conn.cursor()

**Remove Existing Tables**

Any existing versions of the tables are removed before creating the new table structure.

This ensures that the database starts with a clean schema and prevents conflicts with previously created tables.

The changes are committed to the database after the tables are dropped.

In [3]:
cur.execute('DROP TABLE IF EXISTS derived_metrics;')
cur.execute('DROP TABLE IF EXISTS nutrient_info;')
cur.execute('DROP TABLE IF EXISTS product_info;')
conn.commit()

**Create the Product Information Table**

The `product_info` table stores the core identification details of each chocolate product.

The `product_code` is defined as the **primary key**, ensuring that each product can be uniquely identified.

The table contains:

- `product_code` — Unique product identifier
- `product_name` — Name of the chocolate product
- `brand` — Brand associated with the product

In [4]:
cur.execute('DROP TABLE IF EXISTS product_info;')
cur.execute('''
    CREATE TABLE product_info (
    product_code TEXT PRIMARY KEY,
    product_name TEXT,
    brand TEXT
);
''')
conn.commit()

**Create the Nutrient Information Table**

The `nutrient_info` table stores the nutritional characteristics associated with each product.

The `product_code` acts as a foreign key referencing `product_info(product_code)`. This establishes a relationship between product information and its nutritional data.

The table contains energy, carbohydrates, sugars, fats, proteins, fiber, salt, sodium, nutrition score, NOVA group, and the estimated fruits, vegetables, and nuts content.

In [5]:
cur.execute('DROP TABLE IF EXISTS nutrient_info;')
cur.execute('''
   CREATE TABLE nutrient_info (
    product_code TEXT,
    energy_kcal_value FLOAT,
    energy_kj_value FLOAT,
    carbohydrates_value FLOAT,
    sugars_value FLOAT,
    fat_value FLOAT,
    saturated_fat_value FLOAT,
    proteins_value FLOAT,
    fiber_value FLOAT,
    salt_value FLOAT,
    sodium_value FLOAT,
    fruits_vegetables_nuts_estimate_from_ingredients_100g FLOAT,
    nutrition_score_fr INTEGER,
    nova_group INTEGER,
    FOREIGN KEY (product_code) REFERENCES product_info(product_code)
);
''')
conn.commit()

**Create the Derived Metrics Table**

The `derived_metrics` table stores the features created during the feature engineering stage.

These derived features include:

- `sugar_to_carb_ratio`
- `calorie_category`
- `sugar_category`
- `is_ultra_processed`

The `product_code` is used as a foreign key to connect these metrics with the corresponding product in the `product_info` table.

In [6]:
cur.execute('DROP TABLE IF EXISTS derived_metrics;')
cur.execute('''
   CREATE TABLE derived_metrics (
    product_code TEXT,
    sugar_to_carb_ratio FLOAT,
    calorie_category TEXT,
    sugar_category TEXT,
    is_ultra_processed TEXT,
    FOREIGN KEY (product_code) REFERENCES product_info(product_code)
);
''')
conn.commit()

In [7]:
## Load Product Information

## The product-level information from `df_main` is inserted into the `product_info` table.

## Each row contains the product code, product name, and brand. The product code is used as the primary identifier for linking this information with the other relational tables.

for _, row in df_main.iterrows():
    cur.execute("""
        INSERT INTO product_info (product_code, product_name, brand)
        VALUES (%s, %s, %s)
    """, (row['product_code'], row['product_name'], row['brands']))
conn.commit()

In [8]:
## Load Nutritional Information

## The nutritional features from `df_main` are inserted into the `nutrient_info` table.

## Each nutritional record is associated with a product through `product_code`, which maintains the relationship between the nutritional information and the corresponding product.

for _, row in df_main.iterrows():
    cur.execute("""
        INSERT INTO nutrient_info VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row['product_code'],
        row['energy_kcal_value'],
        row['energy_kj_value'],
        row['carbohydrates_value'],
        row['sugars_value'],
        row['fat_value'],
        row['saturated_fat_value'],
        row['proteins_value'],
        row['fiber_value'],
        row['salt_value'],
        row['sodium_value'],
        row['fruits_vegetables_nuts_estimate_from_ingredients_100g'],
        row['nutrition_score_fr'],
        row['nova_group']
    ))
conn.commit()

In [9]:
## Load Derived Metrics

## The engineered features from `df_main` are inserted into the `derived_metrics` table.

## Each record is linked to its corresponding product using `product_code`, allowing the derived metrics to be combined with product and nutritional information during SQL analysis.

for _, row in df_main.iterrows():
    cur.execute("""
        INSERT INTO derived_metrics VALUES (%s, %s, %s, %s, %s)
    """, (
        row['product_code'],
        row['sugar_to_carb_ratio'],
        row['calorie_category'],
        row['sugar_category'],
        row['is_ultra_processed']
    ))
conn.commit()